# Standalone PP true-area parameterization

Generate the continuation sequence from the current UV map using our Tutte initialization. SLIM and shifted projected CM use the existing MeshFEM implementations through `PP_utils`.


In [ ]:
from pathlib import Path
import PP_utils

In [ ]:
PROJECT_ROOT = Path(PP_utils.__file__).resolve().parents[2]
INPUT_MESH = PROJECT_ROOT.parent / 'PP/data/data2/fig3.obj'
INPUT_MESH = Path('../../../Models/PPdata/data1/fig13_c1.obj')

In [ ]:
THREAD_COUNT = 1
MAX_ITER_NUM = 5000  # Includes PP controller passes that take no solver step.
BOUND_DISTORTION_K = 250.0
CONVERGENCE_RATE = 1e-6

In [ ]:
uv, history = PP_utils.run_pp_true_area(
    INPUT_MESH,
    thread_count=THREAD_COUNT,
    max_iter_num=MAX_ITER_NUM,
    bound_distortion_K=BOUND_DISTORTION_K,
    convergence_rate=CONVERGENCE_RATE,
    return_history=True,
)


In [ ]:
{'uv_shape': uv.shape, **history['summary']}


In [ ]:
assert uv.ndim == 2 and uv.shape[1] == 2
assert all(len(history[key]) == history['summary']['sum_iter'] + 1
           for key in ('energy', 'grad_norm', 't_sequence', 'time'))
assert history['time'][0] == history['t_sequence'][0] == 0.0
assert history['summary']['final_min_source_det'] > 0.0
assert (history['active_reference_lambda'][history['stage'] == 'cm_source'] == 1.0).all()


In [ ]:
fig, axes = PP_utils.plot_pp_history(history)


## PP Linux comparison

Automatically match the completed run’s model and thread count in `true_area`. Iteration zero is initialization; cumulative times retain their original clocks (ours: wall time, Linux: CPU time).


In [ ]:
linux_benchmark_dir = 'linux_exp_data/PP_benchmark_Linux_0823/'

In [ ]:
comparison = PP_utils.pp_linux_comparison_widget(history, linux_benchmark_dir)
display(comparison)
